In [1]:
library(SNFtool)

In [2]:
# #### MRNA MCI
mrna_lmci <- read.csv("extdata/all_lmci_mrna.csv", header=TRUE, row.names=1)

# #### Metab EMCI
metab_lmci <- read.csv("extdata/all_lmci_metab.csv", header=TRUE, row.names=1)

mrna_lmci_all <- read.csv("extdata/mrna20k_lmci.csv", header=TRUE)

In [3]:
rownames(mrna_lmci_all) = rownames(metab_lmci)
mrna_lmci_all = mrna_lmci_all[,c(-1,-2)]
mrna_lmci_all = mrna_lmci_all[,-1]
metab_lmci = metab_lmci[,-1]
mrna_lmci  =  mrna_lmci_all

In [4]:
c(dim(mrna_lmci), dim(metab_lmci))

[1]   200 20031   200   172

In [5]:
Dist1 = (dist2(as.matrix(mrna_lmci),as.matrix(mrna_lmci)))^(1/2)
Dist2 = (dist2(as.matrix(metab_lmci),as.matrix(metab_lmci)))^(1/2)

In [6]:
### parameters

## First, set all the parameters:
K = 10;		# number of neighbors, usually (10~30)
alpha = 0.8;  	# hyperparameter, usually (0.3~0.8)
T = 20; 	# Number of Iterations, usually (10~20)

In [7]:
W1 = affinityMatrix(Dist1, K, alpha)
W2 = affinityMatrix(Dist2, K, alpha)

In [8]:
W = SNF(list(W1,W2), K, T)

In [9]:
library("RColorBrewer")
my_palette <- colorRampPalette(c("blue","red"))(n = 500)

In [10]:
#### estimate number of clusters
estimationResult = estimateNumberOfClustersGivenGraph(W, 2:15);
estimationResult

$`Eigen-gap best`
[1] 2

$`Eigen-gap 2nd best`
[1] 5

$`Rotation cost best`
[1] 2

$`Rotation cost 2nd best`
[1] 5

In [11]:
# #### make true labels
truelabel = rep(1,200)


In [12]:
truelabel

[1] 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 [38] 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 [75] 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
[112] 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
[149] 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
[186] 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1

In [13]:
## Get a matrix containing the group information 
## for the samples such as the SpectralClustering result and the True label

C = 2                     # number of clusters
group = spectralClustering(W,C);    # the final subtypes information
group

[1] 2 1 1 2 2 1 2 1 2 1 2 2 2 2 1 1 1 2 2 2 2 1 2 2 2 2 2 2 1 2 2 1 2 2 1 1 2
 [38] 2 2 2 1 2 2 1 1 2 1 2 2 1 2 1 1 1 1 2 1 2 2 2 2 2 2 2 1 1 1 2 1 2 1 1 1 1
 [75] 2 2 1 1 1 1 2 2 2 1 1 2 1 1 1 2 2 2 2 1 1 2 2 1 1 1 1 2 2 2 2 1 1 1 1 2 2
[112] 2 2 2 1 1 2 1 2 1 2 2 1 1 1 2 1 2 2 2 2 2 2 1 1 2 1 1 1 1 1 2 1 1 1 1 1 2
[149] 1 1 2 1 2 2 2 1 1 2 2 1 2 1 1 2 2 1 1 2 1 1 2 2 1 1 2 2 2 2 2 2 1 2 2 2 1
[186] 1 1 1 1 2 1 1 1 1 1 2 2 2 2 2

In [14]:
subtype_info<-read.csv("extdata/cn_emci_lmci_ad_clin_pseudotime.csv",header=TRUE)
table(subtype_info$SNF)
name_lmci1<-subtype_info$PID[which(subtype_info$SNF=="lmcisubtype1")]
name_lmci2<-subtype_info$PID[which(subtype_info$SNF=="lmcisubtype2")]

snflabel=rep(0,200)

snflabel[which(rownames(W)%in% name_lmci1)]=1
snflabel[which(rownames(W)%in% name_lmci2)]=2
snflabel


          AD           CN emcisubtype1 emcisubtype2 lmcisubtype1 lmcisubtype2 
         339          534          108           93           85          115 

[1] 2 1 1 2 2 1 2 1 2 1 2 2 2 2 1 1 1 2 2 2 2 1 1 2 2 2 2 2 2 2 2 1 2 2 1 2 2
 [38] 2 2 1 1 2 2 2 1 2 1 2 2 1 2 2 2 1 1 2 1 2 2 2 2 2 2 2 2 1 1 2 1 2 2 1 1 1
 [75] 2 2 1 1 1 1 2 2 2 1 1 2 1 1 1 2 2 2 2 1 1 2 2 1 1 1 1 2 2 2 2 1 1 1 1 2 2
[112] 2 2 2 1 2 2 1 2 1 2 2 1 1 2 2 1 2 2 1 2 2 2 1 1 2 1 1 1 1 1 2 1 2 1 1 1 2
[149] 1 1 2 1 2 2 2 1 1 2 2 1 2 1 1 2 2 1 1 2 2 2 2 2 1 1 2 2 2 2 2 2 1 2 2 2 1
[186] 1 1 1 1 2 1 2 1 1 1 2 2 2 2 2

In [15]:
M_label=cbind(snflabel,truelabel)
colnames(M_label)=c("spectralClustering","TrueLabel")

M_label_colors=t(apply(M_label,1,getColorsForGroups))
## or choose you own colors for each label, for example:
M_label_colors=cbind("spectralClustering"=getColorsForGroups(M_label[,"spectralClustering"],colors=c("deepskyblue1","seagreen2","orange1")), ### 
"TrueLabel"=getColorsForGroups(M_label[,"TrueLabel"], colors=c("navy", "red1"))) 
#"TrueLabel"=getColorsForGroups(M_label[,"TrueLabel"], colors=c("khaki1"))) ### in the same order as ()
png(filename="figs/RP_2f.png", width=3.25,  height= 3.25, units= "in", res = 600)
displayClustersWithHeatmap(W,snflabel, M_label_colors)
dev.off()

NULL

png 
  2

In [16]:
### cluster labels for W
paste(group, collapse=",")

[1] "2,1,1,2,2,1,2,1,2,1,2,2,2,2,1,1,1,2,2,2,2,1,2,2,2,2,2,2,1,2,2,1,2,2,1,1,2,2,2,2,1,2,2,1,1,2,1,2,2,1,2,1,1,1,1,2,1,2,2,2,2,2,2,2,1,1,1,2,1,2,1,1,1,1,2,2,1,1,1,1,2,2,2,1,1,2,1,1,1,2,2,2,2,1,1,2,2,1,1,1,1,2,2,2,2,1,1,1,1,2,2,2,2,2,1,1,2,1,2,1,2,2,1,1,1,2,1,2,2,2,2,2,2,1,1,2,1,1,1,1,1,2,1,1,1,1,1,2,1,1,2,1,2,2,2,1,1,2,2,1,2,1,1,2,2,1,1,2,1,1,2,2,1,1,2,2,2,2,2,2,1,2,2,2,1,1,1,1,1,2,1,1,1,1,1,2,2,2,2,2"

In [17]:
##### get indexes
subtype1 = c(which(group==1))

subtype2 = c(which(group==2))


In [18]:
paste(subtype1, collapse=",")

[1] "2,3,6,8,10,15,16,17,22,29,32,35,36,41,44,45,47,50,52,53,54,55,57,65,66,67,69,71,72,73,74,77,78,79,80,84,85,87,88,89,94,95,98,99,100,101,106,107,108,109,115,116,118,120,123,124,125,127,134,135,137,138,139,140,141,143,144,145,146,147,149,150,152,156,157,160,162,163,166,167,169,170,173,174,181,185,186,187,188,189,191,192,193,194,195"

In [19]:
paste(subtype2, collapse=",")

[1] "1,4,5,7,9,11,12,13,14,18,19,20,21,23,24,25,26,27,28,30,31,33,34,37,38,39,40,42,43,46,48,49,51,56,58,59,60,61,62,63,64,68,70,75,76,81,82,83,86,90,91,92,93,96,97,102,103,104,105,110,111,112,113,114,117,119,121,122,126,128,129,130,131,132,133,136,142,148,151,153,154,155,158,159,161,164,165,168,171,172,175,176,177,178,179,180,182,183,184,190,196,197,198,199,200"

In [20]:
c(length(subtype1),length(subtype2))

[1]  95 105